# Reproducible adult screening-label evaluation
Read `research/STUDY.md` for source verification, label limitations, and the evaluation protocol. This notebook runs the same evaluation as `python -m src.evaluate`; serving models are not overwritten. The AQ >=6 referral rule and the empirical >=7 dataset rule are distinct.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/evaluate.py').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.evaluate import evaluate
import pandas as pd


In [ ]:
import json
protocol = json.loads((ROOT / 'research/protocol.json').read_text())
audit = evaluate(ROOT / 'data/train.csv', ROOT / 'research/evaluation', protocol['seeds'], protocol['folds'], protocol['bootstrap_draws'], ROOT / 'research/sources/uci_adult_426.zip', protocol['tree_depths'])
audit


In [ ]:
metrics = pd.read_csv(ROOT / 'research/evaluation/pooled_metrics.csv')
metrics.groupby(['population', 'method'])[['precision', 'recall', 'f1', 'roc_auc', 'accuracy']].agg(['mean', 'std'])


In [ ]:
pd.read_csv(ROOT / 'research/evaluation/conditional_bootstrap.csv')


# Why is agreement so high?
The following descriptive analyses explain screening-label reconstruction. The threshold scan uses all labels to describe their relationship with scores; it does not select a clinical cutoff or provide independent validation.


In [ ]:
OUT = ROOT / 'research/evaluation'
label_table = pd.read_csv(OUT / 'label_by_score.csv')
scan = pd.read_csv(OUT / 'threshold_scan.csv')
display(label_table.loc[label_table.population == 'primary'])
display(scan.loc[scan.population == 'primary', ['threshold', 'disagreements', 'accuracy', 'precision', 'recall', 'f1', 'tn', 'fp', 'fn', 'tp']])


## Questionnaire-only logistic coefficients
Weights come from training folds, after fold-local scaling and oversampling. Raw-item coefficients describe a 0-to-1 item change holding other items fixed. These weights reproduce dataset labels; they are not clinical effect sizes. Across-fold/repeat variability is descriptive, not a confidence interval.


In [ ]:
coef = pd.read_csv(OUT / 'questionnaire_logistic_coefficients.csv')
coef.loc[coef.population == 'primary'].groupby('item')[['standardized_coefficient', 'raw_item_coefficient', 'one_point_odds_ratio']].agg(['mean', 'std', 'min', 'max'])


## Decision-tree depth comparison
Use the same grouped folds, seeds, and oversampling as the main experiment. Compare individual questionnaire items with combined features including their sum. Minimum leaf size stays fixed at 10 for every depth; unlimited depth does not remove that constraint. This is a descriptive capacity comparison, not validation-based hyperparameter selection.


In [ ]:
depth = pd.read_csv(OUT / 'tree_depth_metrics.csv')
display(depth.loc[depth.population == 'primary'].groupby(['feature_set', 'max_depth'])[['accuracy', 'precision', 'recall', 'f1', 'roc_auc']].agg(['mean', 'std']))
roots = pd.read_csv(OUT / 'tree_depth_fold_metrics.csv')
stumps = roots.loc[(roots.population == 'primary') & (roots.max_depth.astype(str) == '1')]
display(stumps[['seed', 'feature_set', 'root_feature', 'root_threshold_raw', 'training_accuracy', 'accuracy']])


## Interpretation
Removing the sum alone leaves all ten scored items, which can reproduce the sum. Compare the non-questionnaire feature set before claiming independence from AQ information. Bootstrap intervals are conditional on fixed predictions; perfect reconstruction can produce degenerate intervals. These are screening-label comparisons, not diagnostic validation.
